<img src="./Imagenes/logo_UTN.svg" align="right" width="150" /> 

#### Teoría de los Circuitos II

# Trabajo semanal 1
#### Lucas Gallero
----

# Enunciado:

Dado el siguiente circuito:

<div align="center">
    <img src="./Imagenes/enunciado.png" alt="Enunciado" width="600"/>
</div>

1. Obtener analíticamente la función transferencia $T(s)=\frac{V_2(s)}{V_1(s)}$. Realizar diagramas de módulo, fase y de polos y ceros para el caso en el que$\frac{R_2}{R_1}=1$.

2. ¿De qué tipo de filtro se trata? Investigue y comente brevemente qué utilidad podría tener este tipo de circuitos.

3. Proponga una norma de impedancia $\Omega_z$ y frecuencia $\Omega_{\omega}$ de forma tal de llegar a una transferencia **normalizada** (escalada en impedancia y frecuencia).

4. Simule la función transferencia normalizada en Python.

5. Simule la red normalizada en LTspice, y obtenga su respuesta en frecuencia.

**Bonus:**

- +1 💎 Obtener una RED normalizada que responda a la función hallada en 2).
- +1 🧠 Verifique los resultados de 1 y 2 mediante el módulo de simulación simbólica **SymPy**.
- +1 🤯 Analice similitudes y diferencias con ambas redes del ejercicio 07 del apartado **"Amplificadores OPAMP y OTA"** de la guía de TPs.

In [9]:
# PyTC2: La librería para TC2
from pytc2.sistemas_lineales import pzmap, GroupDelay, bodePlot

from scipy.signal import TransferFunction
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.abc import s


# las librerías que usarremos las cargamos solo una vez.


# Resolución


### 1.1 Obtención Función Transferencia

Para obtener la función transferencia del circuito se aplicó el método sistemático por nodos, suponiendo al amplificador operacional ideal y trabajando en régimen lineal. Bajo estas condiciones se cumple que:

* La corriente de entrada en ambos terminales del AO es nula,
* La tensión en la entrada inversora y no inversora es la misma.

Se define entonces una tensión de nodo común:
$$
V_T = V^+ = V^-
$$

Para el análisis mediante el método de nodos, se tomarán como nodos de estudio los correspondientes a las entradas inversora y no inversora del amplificador operacional.


$$
\text{(V-)} \qquad
V_T\left(\frac{1}{R_1}+\frac{1}{R_2}\right)-\frac{V_1}{R_1}-\frac{V_2}{R_2}=0
\tag{1}
$$


$$
\text{(V+)} \qquad
V_T\left(
sC+\frac{1}{R_3}\right)-V_1sC=0
\tag{2}
$$

**Despejo $V_T$ de (2):**

$$
V_T\left(\frac{sCR_3+1}{R_3}\right)=V_1sC
$$

$$
\boxed{V_T = V_1\left(\frac{sCR_3}{sCR_3+1}\right)}
\tag{3}
$$



**Reemplazo (3) en (1):**

$$
\left( V_1 \frac{sC_1R_3}{1+sC_1R_3} \right)\left(\frac{1}{R_1}+\frac{1}{R_2}\right)-\frac{V_1}{R_1}-\frac{V_2}{R_2}=0
$$

$$
\left( \frac{sC_1R_3}{1+sC_1R_3} \right)\left(\frac{1}{R_1}+\frac{1}{R_2}\right)-\frac{1}{R_1}-\frac{V_2}{V_1R_2}=0
$$

$$
\frac{V_2}{V_1R_2}=\left( \frac{sC_1R_3}{1+sC_1R_3} \right)\left(\frac{1}{R_1}+\frac{1}{R_2}\right)-\frac{1}{R_1}
$$

$$
\frac{V_2}{V_1}=R_2\left[\left( \frac{sC_1R_3}{1+sC_1R_3} \right)\left(\frac{1}{R_1}+\frac{1}{R_2}\right)-\frac{1}{R_1}\right]
$$

$$
\frac{V_2}{V_1}= \left( \frac{sC_1R_3}{1+sC_1R_3} \right)\left(\frac{R_2}{R_1}+1\right)-\frac{R_2}{R_1}
$$

$$
\frac{V_2}{V_1}= \frac{sC_1R_3\left(\frac{R_2}{R_1}+1\right)-\frac{R_2}{R_1}(1+sC_1R_3)}{1+sC_1R_3}
$$

$$
\frac{V_2}{V_1}= \frac{sC_1R_3-\frac{R_2}{R_1}}{1+sC_1R_3}
$$

$$
\boxed{T(s) =\frac{V_2}{V_1}= \frac{s-\frac{R_2}{R_1R_3C1}}{s+\frac{1}{R_3C1}}}
$$


#### Verificación de Función Transferencia por Simulación

In [11]:
V1, V2, Vx = sp.symbols("V1, V2, Vx")
Y1, Y2, Y3, YC, Y, C1, R1, R2, R3 = sp.symbols("Y1, Y2, Y3, YC, Y, C1, R1, R2, R3")

sistema = sp.solve([ 
                    Vx*(Y1+Y2)-V1*Y1-V2*Y2,
                    Vx*(YC+Y3)-V1*YC
            ], 
            [V1,V2,Vx])

transf_func = sistema[V2]/sistema[V1]
transf_func = sp.simplify(transf_func) 

tf = transf_func.subs(Y1,1/R1)
tf = tf.subs(Y2,1/R2)
tf = tf.subs(Y3,1/R3)
tf = tf.subs(YC,s*C1)
num = tf*(s*C1+1/R3)
den = num/tf
num = sp.monic(num)
den = sp.monic(den)
num/den

(C1*s - R2/(R1*R3))/(C1*s + 1/R3)

Reordenando algebraicamente los términos obtenidos, la función transferencia del circuito queda expresada como:
$$
\boxed{T(s) = \frac{s-\frac{R_2}{R_1R_3C1}}{s+\frac{1}{R_3C1}}}
$$

Por lo tanto, se verifica que la expresión obtenida para la transferencia es consistente con el análisis nodal realizado.

Bajo la condición $\frac{R_2}{R_1}=1$, la función transferencia obtenida adopta la forma:
$$\boxed{T(s)=\frac{s-\frac{1}{R_3C_1}}{s+\frac{1}{R_3C_1}}}$$

---


### 1.2 Diagrama de Modulo, Fase, Polos y Ceros

#### Análisis del módulo
Para analizar el módulo, se evalúa la transferencia sobre el eje imaginario, es decir, tomando:
$$
s=j\omega
$$

Entonces:

$$
T(j\omega)=\frac{j\omega-\frac{1}{R_3C_1}}{j\omega+\frac{1}{R_3C_1}}
$$

El módulo resulta:

$$
|T(j\omega)|=\left|\frac{j\omega-\frac{1}{R_3C_1}}{j\omega+\frac{1}{R_3C_1}}\right|
$$

Elevando al cuadrado:

$$
|T(j\omega)|^2=
\frac{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
$$

Por lo tanto:

$$
|T(j\omega)|^2=1
$$

y finalmente:

$$
\boxed{|T(j\omega)|=1}
$$

#### Análisis de la fase
Partiendo de:

$$
T(j\omega)=\frac{j\omega-\frac{1}{R_3C_1}}{j\omega+\frac{1}{R_3C_1}}
$$

Para obtener la fase, se multiplica por el conjugado del denominador:

$$
T(j\omega)=
\frac{j\omega-\frac{1}{R_3C_1}}{j\omega+\frac{1}{R_3C_1}}
\cdot
\frac{\frac{1}{R_3C_1}-j\omega}{\frac{1}{R_3C_1}-j\omega}
$$

Desarrollando:

$$
T(j\omega)=
\frac{\left(j\omega-\frac{1}{R_3C_1}\right)^2}
{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
$$

Expandiendo el numerador:

$$
T(j\omega)=
\frac{-\omega^2+\left(\frac{1}{R_3C_1}\right)^2-\frac{2j\omega}{R_3C_1}}
{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
$$

Por lo tanto:

$$
\Re\{T(j\omega)\}=
\frac{\left(\frac{1}{R_3C_1}\right)^2-\omega^2}
{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
$$

$$
\Im\{T(j\omega)\}=
\frac{-\frac{2\omega}{R_3C_1}}
{\omega^2+\left(\frac{1}{R_3C_1}\right)^2}
$$

Entonces, la fase resulta:

$$
\phi(\omega)=\arctan\left(
\frac{\Im\{T(j\omega)\}}{\Re\{T(j\omega)\}}
\right)
$$

$$
\phi(\omega)=\arctan\left(
\frac{-\frac{2\omega}{R_3C_1}}
{\left(\frac{1}{R_3C_1}\right)^2-\omega^2}
\right)
$$

También puede escribirse como:

$$
\boxed{
\phi(\omega)=\arctan\left(
\frac{-2\omega R_3C_1}
{1-\omega^2R_3^2C_1^2}
\right)}
$$